## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 02: Feature Engineering and F&O Construction**

---

**Scope:** Computes technical and range-based volatility features on index OHLCV, and parses the raw Bhavcopy archive into lot-size-invariant daily turnover and open-interest panels for Bank Nifty and Nifty 50.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **02_feature_engineering_and_fo_construction.ipynb** <- *you are here*


### 1. Technical and volatility features

`src.features.add_all_features` adds RSI, MACD, Bollinger Bands, Average True Range, close-to-close rolling volatility, and the Parkinson and Garman-Klass range-based realized volatility estimators (used later in the interrupted-time-series check), plus the cyclical expiry-distance encoding.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
features = pd.read_parquet(config.INTERIM_DATA_DIR / 'banknifty_features.parquet')
print(f'{len(features)} rows, {features["Date"].min()} to {features["Date"].max()}')
features[['Date', 'Close', 'Return', 'RSI', 'Parkinson_Vol_1d', 'GarmanKlass_Vol_1d',
          'Days_To_Expiry', 'Expiry_Week']].tail(10)

1680 rows, 2020-01-01 to 2026-10-16


,Date,Close,Return,RSI,Parkinson_Vol_1d,GarmanKlass_Vol_1d,Days_To_Expiry,Expiry_Week
1670,2026-10-05,54714.101562,0.004825,36.311878,0.009012,0.010513,16,0
1671,2026-10-06,55128.398438,0.007544,41.495709,0.003998,0.003947,15,0
1672,2026-10-07,55055.550781,-0.001322,40.865852,0.007694,0.009006,14,0
1673,2026-10-08,54515.050781,-0.009866,36.445589,0.007243,0.006071,13,0
1674,2026-10-09,55256.648438,0.013512,45.203452,0.008756,0.008545,12,0
1675,2026-10-12,54918.000000,-0.006148,42.334568,0.007718,0.007672,11,0
1676,2026-10-13,55481.050781,0.010200,48.218883,0.008885,0.008230,10,0
1677,2026-10-14,55316.000000,-0.002979,46.714069,0.005165,0.005995,9,0
1678,2026-10-15,55810.500000,0.008900,51.588742,0.008484,0.009121,8,0
1679,2026-10-16,55844.000000,0.000600,51.909704,0.004990,0.005802,7,0


### 2. Lot-size-invariant F&O turnover and open interest

`scripts.build_fo_volume_oi_dataset` and `scripts.build_nifty_volume_oi_dataset` parse the raw Bhavcopy archive (both formats) into one daily row per index, with total traded value (rupees, lot-size-invariant) and lot-size-adjusted notional open interest alongside the raw contract-count figures. This is a one-time parse of ~1,700 files per index; rebuild with:

```bash
python -m scripts.build_fo_volume_oi_dataset
python -m scripts.build_nifty_volume_oi_dataset
```

The cell below loads the already-built panels.

In [3]:
bn_fo = pd.read_parquet(config.INTERIM_DATA_DIR / 'banknifty_fo_volume_oi.parquet')
nf_fo = pd.read_parquet(config.INTERIM_DATA_DIR / 'nifty_fo_volume_oi.parquet')
print(f'Bank Nifty F&O panel: {len(bn_fo)} days, {bn_fo["Date"].min()} to {bn_fo["Date"].max()}')
print(f'Nifty 50 F&O panel: {len(nf_fo)} days')
bn_fo[['Date', 'Total_Volume', 'Total_Value', 'Total_OpenInterest', 'Total_OpenInterest_Notional', 'LotSize']].tail(10)

Bank Nifty F&O panel: 1679 days, 2020-01-01 00:00:00 to 2026-10-16 00:00:00
Nifty 50 F&O panel: 1679 days


,Date,Total_Volume,Total_Value,Total_OpenInterest,Total_OpenInterest_Notional,LotSize
1669,2026-10-05,2115580.0,3.523219e+12,33543570.0,1.006307e+09,30.0
1670,2026-10-06,1496318.0,2.498900e+12,34109190.0,1.023276e+09,30.0
1671,2026-10-07,2839776.0,4.740666e+12,35696730.0,1.070902e+09,30.0
1672,2026-10-08,2109173.0,3.500711e+12,36141480.0,1.084244e+09,30.0
1673,2026-10-09,2089760.0,3.490479e+12,36889860.0,1.106696e+09,30.0
1674,2026-10-12,1954985.0,3.259394e+12,38039730.0,1.141192e+09,30.0
1675,2026-10-13,2467758.0,4.126991e+12,37762890.0,1.132887e+09,30.0
1676,2026-10-14,2265940.0,3.797883e+12,37685940.0,1.130578e+09,30.0
1677,2026-10-15,2672916.0,4.497377e+12,38008440.0,1.140253e+09,30.0
1678,2026-10-16,2745965.0,4.624021e+12,39326670.0,1.179800e+09,30.0
